# Used Cars Exploratory Data Analysis

**Dataset:** Car Details from CarDekho (4,340 rows, 8 columns). The notebook keeps categories readable and removes only exact duplicate rows. It does not clip valid price, year, or mileage values.

## 1. Imports and data loading
Upload the CSV when prompted. You can obtain the matching file from the dataset source linked in the report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
csv_file = next(iter(uploaded))
df_raw = pd.read_csv(csv_file)
df_raw.head()

## 2. Overview, types, missing values, and categories

In [ ]:
print('Shape:', df_raw.shape)
df_raw.info()
display(df_raw.describe(include='all').T)
print('Missing values per column:\n', df_raw.isna().sum())
print('Exact duplicate rows:', df_raw.duplicated().sum())
for col in ['fuel', 'seller_type', 'transmission', 'owner']:
    print(f'\n{col}:\n{df_raw[col].value_counts(dropna=False)}')

## 3. Cleaning
The source has repeated identical rows. Remove exact duplicates so repeated copies do not inflate counts. No missing values are present in this file, so no imputation is needed. Keep the original values for valid rare cars and high prices; flag potential outliers for interpretation rather than clipping them.

In [ ]:
df = df_raw.drop_duplicates().copy()
print('Before cleaning:', df_raw.shape)
print('After removing exact duplicates:', len(df))
print('Rows removed:', len(df_raw) - len(df))
print('Missing values after cleaning:', int(df.isna().sum().sum()))
df.head()

## 4. Descriptive statistics

In [ ]:
display(df[['year', 'selling_price', 'km_driven']].describe().T)
print('Median selling price (INR):', df['selling_price'].median())
print('Median kilometres driven:', df['km_driven'].median())

## 5. Univariate analysis (bar charts, histograms, box plot)

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(8,4)); sns.countplot(data=df, x='fuel', order=df['fuel'].value_counts().index); plt.title('Listings by fuel type'); plt.xlabel('Fuel type'); plt.ylabel('Listings'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); sns.histplot(df['selling_price']/100000, bins=35); plt.title('Selling price distribution'); plt.xlabel('Selling price (lakh INR)'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); sns.histplot(df['year'], discrete=True); plt.title('Vehicle year distribution'); plt.xlabel('Year'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,3)); sns.boxplot(x=df['selling_price']/100000); plt.title('Selling price spread and potential outliers'); plt.xlabel('Selling price (lakh INR)'); plt.tight_layout(); plt.show()

**Chart observations:** Diesel is the most common fuel (50.3%); selling prices have a long upper tail (mean INR 473,913, median INR 350,000); vehicle years span 1992-2020 and cluster in recent years; the box plot flags high prices for review rather than automatic deletion.

## 6. Bivariate analysis (scatter plot and grouped summaries)

In [ ]:
sample = df.sample(min(1500, len(df)), random_state=42)
plt.figure(figsize=(8,5)); sns.scatterplot(data=sample, x='km_driven', y='selling_price', hue='transmission', alpha=.6); plt.title('Mileage and selling price (sample)'); plt.xlabel('Kilometres driven'); plt.ylabel('Selling price (INR)'); plt.tight_layout(); plt.show()
print('Selling price vs km driven correlation:', df[['selling_price','km_driven']].corr().iloc[0,1].round(3))
print('Median selling price by fuel type:\n', df.groupby('fuel')['selling_price'].median().sort_values(ascending=False))

**Observation:** Selling price and kilometres driven have correlation -0.19; there is a weak negative association with substantial variation, so mileage alone does not explain price.

## 7. Multivariate analysis (stacked bar)

In [ ]:
pd.crosstab(df['fuel'], df['transmission']).plot(kind='bar', stacked=True, figsize=(8,4)); plt.title('Fuel type by transmission'); plt.xlabel('Fuel type'); plt.ylabel('Listings'); plt.tight_layout(); plt.show()
print('Listings by fuel, transmission, and seller:')
display(pd.crosstab([df['fuel'], df['transmission']], df['seller_type']))

**Observation:** Manual listings account for 91.3% of the cleaned data. The stacked chart shows how fuel and transmission counts vary together; very small fuel groups should be treated cautiously.

## 8. Outlier review and evidence-based insights
An IQR rule can flag unusually high or low values, but a flag is not proof of an error. Do not clip the `year` column: years are discrete calendar values and valid older vehicles should remain. Review high prices and mileage against the record context.

In [ ]:
for col in ['selling_price', 'km_driven']:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    print(col, 'IQR bounds:', low, high, '| flagged rows:', int(((df[col] < low) | (df[col] > high)).sum()))
print('Exact duplicate rows removed:', int(df_raw.duplicated().sum()))
print('Missing values:', int(df_raw.isna().sum().sum()))
print('Correlation selling price / km driven:', round(df[['selling_price','km_driven']].corr().iloc[0,1], 3))

## 9. Conclusion and limitations
This is a set of used-car listings, not a complete market census. Prices are influenced by omitted factors such as condition, city, trim details, and listing date. The dataset snapshot is dated and should not be treated as current market pricing. Associations in this EDA do not establish cause and effect.